In [1]:
# Core libraries
import pandas as pd #data manipulation
import numpy as np #numerical operations

# Visualization
import matplotlib.pyplot as plt #plotting
import seaborn as sns #statistical visualization

# Model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

# Model
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("../data/processed/tmdb_clean.csv")

print("Dataset shape:", df.shape)

Dataset shape: (16718, 26)


In [12]:
# Target
y = df["profitable"].astype(int)

# Date features
df["release_year"] = pd.to_datetime(df["release_date"]).dt.year
df["release_month"] = pd.to_datetime(df["release_date"]).dt.month

# Seasonal flags
df["is_summer"] = df["release_month"].isin([5,6,7]).astype(int)
df["is_holiday"] = df["release_month"].isin([11,12]).astype(int)

# Log transform budget
df["log_budget"] = np.log1p(df["budget"])

# Runtime bucket
df["long_movie"] = (df["runtime"] > 120).astype(int)

# Numerical features only (same as strict model)
numerical_features = [
    "log_budget",
    "runtime",
    "release_year",
    "is_summer",
    "is_holiday",
    "long_movie"
]

X = df[numerical_features]
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print("X shape:", X.shape)
print("Class balance:\n", y.value_counts())

X shape: (16718, 6)
Class balance:
 profitable
0    10907
1     5811
Name: count, dtype: int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (13374, 6)
Test shape: (3344, 6)


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features)
    ]
)

In [15]:
print("Any NaNs in X_train?")
print(X_train.isnull().sum())

print("\nAny infinite values?")
print(np.isinf(X_train).sum())

Any NaNs in X_train?
log_budget      0
runtime         0
release_year    0
is_summer       0
is_holiday      0
long_movie      0
dtype: int64

Any infinite values?
log_budget      0
runtime         0
release_year    0
is_summer       0
is_holiday      0
long_movie      0
dtype: int64


In [16]:
#initialize logistic regression classifier
#max_iter increased to ensure convergence
#class_weight ="balanced" compensates for class imbalance
#solver = 'lbfgs' work well for small/medium datasets

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs"
)

#build full pipeline (preprocessing + model)
pipeline = Pipeline([
    ("preprocessor", preprocessor), #scale numerical features
    ("model", model) #logistic regression classifcation
])

pipeline.fit(X_train, y_train)

print("Logistic Regression model fit complete.")

Logistic Regression model fit complete.


In [17]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("CV ROC-AUC:", cv_auc.mean())

CV ROC-AUC: 0.5826164744325297


In [18]:
y_val_proba = pipeline.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_val_proba)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)

Best Threshold: 0.42106550270361204


In [19]:
y_test_proba = pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

print("TEST Accuracy:", accuracy_score(y_test, y_test_pred))
print("TEST Precision:", precision_score(y_test, y_test_pred))
print("TEST Recall:", recall_score(y_test, y_test_pred))
print("TEST F1:", f1_score(y_test, y_test_pred))
print("TEST ROC-AUC:", roc_auc_score(y_test, y_test_proba))

TEST Accuracy: 0.4138755980861244
TEST Precision: 0.36354309165526677
TEST Recall: 0.9148020654044751
TEST F1: 0.5203132648066568
TEST ROC-AUC: 0.6045881180871187
